# 🎵 LyricInsight Emotion V3 Training

K-Pop 가사 기반 감정 분석 모델 학습 노트북

**데이터**: GPT API로 레이블링된 K-Pop 가사 (500샘플)

---

## 1️⃣ GPU 확인 및 라이브러리 설치

In [ ]:
# GPU 확인
!nvidia-smi

# 필수 라이브러리 설치
!pip install -q transformers datasets scikit-learn accelerate

## 2️⃣ Google Drive 마운트 및 데이터 업로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 작업 디렉토리 설정
import os
WORK_DIR = '/content/drive/MyDrive/LyricInsight'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

## 3️⃣ 학습 데이터 업로드

아래 3개 파일을 Google Drive의 `LyricInsight/data/` 폴더에 업로드하세요:
- `train.jsonl`
- `val.jsonl`
- `labels.json`

파일 위치: `d:/LyricInsight/data/processed_kpop/`

In [ ]:
# 데이터 폴더 생성
DATA_DIR = f'{WORK_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)

# 파일 확인
import os
print("📂 데이터 파일 확인:")
for f in ['train.jsonl', 'val.jsonl', 'labels.json']:
    path = f"{DATA_DIR}/{f}"
    if os.path.exists(path):
        print(f"  ✅ {f}")
    else:
        print(f"  ❌ {f} - 파일을 업로드해주세요!")

## 4️⃣ 데이터 로드

In [ ]:
import json
from pathlib import Path
import numpy as np
from datasets import Dataset

DATA_DIR = Path(f'{WORK_DIR}/data')

def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

# 레이블 로드
label_names = json.loads((DATA_DIR / 'labels.json').read_text(encoding='utf-8'))
print(f"✅ Labels: {len(label_names)}개")
print(f"   {label_names[:5]}...")

# 데이터 로드
train_rows = load_jsonl(DATA_DIR / 'train.jsonl')
val_rows = load_jsonl(DATA_DIR / 'val.jsonl')
print(f"\n✅ Train: {len(train_rows)}개")
print(f"✅ Val: {len(val_rows)}개")

## 5️⃣ 모델 및 토크나이저 준비

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = 'klue/roberta-base'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    problem_type='multi_label_classification'
)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   num_labels: {len(label_names)}")

## 6️⃣ 데이터셋 전처리

In [ ]:
train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

def cast_labels(batch):
    batch['labels'] = np.array(batch['labels'], dtype=np.float32)
    return batch

train_ds = train_ds.map(cast_labels)
val_ds = val_ds.map(cast_labels)

train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
val_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print("✅ 데이터셋 전처리 완료")

## 7️⃣ 학습 설정 및 실행

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import f1_score

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)
    preds = (probs >= 0.3).astype(int)
    
    micro = f1_score(labels, preds, average='micro', zero_division=0)
    macro = f1_score(labels, preds, average='macro', zero_division=0)
    
    return {'f1_micro': micro, 'f1_macro': macro}

OUT_DIR = f'{WORK_DIR}/models/emotion_v3'

args = TrainingArguments(
    output_dir=OUT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=3e-5,
    per_device_train_batch_size=16,  # GPU 메모리에 맞게 조절
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='f1_micro',
    save_total_limit=2,
    fp16=True,  # GPU 가속
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("🚀 학습 시작!")
trainer.train()

## 8️⃣ 모델 저장

In [ ]:
# 모델 저장
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

# 레이블 정보 저장
with open(f'{OUT_DIR}/labels.json', 'w', encoding='utf-8') as f:
    json.dump(label_names, f, ensure_ascii=False, indent=2)

print(f"✅ 모델 저장 완료: {OUT_DIR}")
print("\n📦 저장된 파일:")
for f in os.listdir(OUT_DIR):
    print(f"   - {f}")

## 9️⃣ 모델 테스트

In [ ]:
import torch

# 테스트 텍스트
test_texts = [
    "너를 만나 행복해",
    "이별이 너무 슬퍼",
    "내일이 기대돼",
    "화가 나서 미치겠어"
]

model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

print("🧪 모델 테스트 결과:\n")
for text in test_texts:
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]
    
    # Top 3 감정
    top_idx = probs.argsort()[::-1][:3]
    top_emotions = [(label_names[i], probs[i]) for i in top_idx]
    
    print(f"📝 \"{text}\"")
    for emo, score in top_emotions:
        print(f"   → {emo}: {score:.2%}")
    print()

## 🔟 모델 다운로드 (ZIP)

In [ ]:
import shutil

# ZIP 파일 생성
zip_path = f'{WORK_DIR}/emotion_v3'
shutil.make_archive(zip_path, 'zip', OUT_DIR)
print(f"✅ ZIP 생성 완료: {zip_path}.zip")

# Colab에서 다운로드
from google.colab import files
files.download(f'{zip_path}.zip')